# 02 - Segmentació de Caràcters (dues passades)

## Per què minAreaRect global no funciona

L'enfocament anterior aplicava `cv2.minAreaRect` sobre **tots** els píxels de primer pla de la imatge binaritzada. El problema és que la núvol de punts global (caràcters + marges + soroll del fons) tendeix a alinear-se amb els eixos de la imatge, de manera que `minAreaRect` retorna quasi sempre un angle ≈ 0° i la "correcció" no fa res.

## Solució: estimar l'angle a partir dels caràcters, no del núvol global

Els caràcters d'una matrícula estan alineats horitzontalment **per disseny**: la seva línia de centroides revela directament la inclinació de la placa. Si ajustem una recta als centroides dels caràcters, el seu pendent `m` ens dóna l'angle real d'inclinació:

$$\alpha = \arctan(m) \cdot \frac{180}{\pi}$$

on $m$ és el pendent de la recta que millor ajusta els centroides dels caràcters (regressió lineal mínims quadrats).

## Esquema de dues passades

| Passada | Mode | Objectiu | Toleràncies |
|---------|------|----------|-------------|
| **1 (tolerant)** | `strict=False` | Estimar l'angle | Alçada ±30%, AR 0.05–1.2 |
| **2 (estricte)** | `strict=True` | Segmentar caràcters finals | Alçada ±15%, AR 0.05–0.95 |

La passada 1 és deliberadament permissiva: accepta alguns bboxes dubtosos perquè l'únic que necessita és prou caràcters per estimar l'angle. La passada 2, sobre la imatge ja alineada, aplica els filtres estrictes que garanteixen qualitat.

**Flux complet per crop:**
```
crop_bgr
  → [Passada 1] gray → thresh → detect_char_bboxes(strict=False) → estimate_skew_angle
  → rotate_image(angle)
  → [Passada 2] gray → thresh → detect_char_bboxes(strict=True) → validació [5-8]
  → resize 28×28 → guardar
```

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re
import os


def mostrar_imatge(titol, imatge, cmap=None):
    plt.figure(figsize=(10, 4))
    plt.title(titol)
    if cmap:
        plt.imshow(imatge, cmap=cmap)
    else:
        plt.imshow(cv2.cvtColor(imatge, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()


# ── Directoris ────────────────────────────────────────────────────────────────
PROCESSED_DIR = Path('../data/processed')   # crops del VJ: {stem}_cand{n}.jpg
OUTPUT_DIR    = Path('../data/chars')       # sortida: {stem}_cand{n}_char{i}.png

# ── Rang de caràcters vàlids ──────────────────────────────────────────────────
N_CHARS_MIN = 5
N_CHARS_MAX = 9

# ── Límit de seguretat per a l'angle estimat ─────────────────────────────────
ALIGN_ANGLE_MAX = 15.0   # graus: si |angle| supera aquest valor no rotem

# ── Mida d'entrada per a la CNN ───────────────────────────────────────────────
CNN_INPUT_W = 28
CNN_INPUT_H = 28

# ── Paràmetres del threshold adaptatiu (idèntics al notebook de referència) ───
ADAPTIVE_BLOCK = 31
ADAPTIVE_C     = 15

# ── Nombre de crops a mostrar a la secció de debug ───────────────────────────
N_DEBUG = 70

EXAMPLE_INDEX = 0   # índex de crop a mostrar com a exemple a la demo


def _glob_crops(directory):
    """Retorna tots els crops disponibles (_box*.png)."""
    files = sorted(
        list(directory.glob('*_box*.png'))
    )
    return files


print("Llibreries carregades.")
print(f"Directori d'entrada : {PROCESSED_DIR}")
print(f"Directori de sortida: {OUTPUT_DIR}")
print(f"Rang de caràcters   : [{N_CHARS_MIN}, {N_CHARS_MAX}]")
print(f"Angle màxim         : ±{ALIGN_ANGLE_MAX}°")
print(f"Crops disponibles   : {len(_glob_crops(PROCESSED_DIR))}")

## Funcions del pipeline de dues passades

### `detect_char_bboxes(thresh, strict)`
Detecta bounding boxes de caràcters sobre una imatge binaritzada. El flag `strict` controla la tolerància dels filtres:
- `strict=False` (passada 1): filtres tous per capturar prou caràcters i estimar l'angle.
- `strict=True` (passada 2): filtres estrictes per a la segmentació final de qualitat.

### `estimate_skew_angle(bboxes)`
Calcula l'angle d'inclinació de la placa per regressió lineal sobre els centroides dels caràcters detectats. L'equació és:

$$\alpha = \arctan(m) \cdot \frac{180}{\pi}$$

on $m$ és el pendent de la recta $c_y = m \cdot c_x + b$ ajustada per mínims quadrats als centroides $(c_x, c_y)$ de cada bbox. Com que els caràcters d'una matrícula estan alineats horitzontalment per disseny, la seva línia de centroides revela directament la inclinació de la placa.

### `rotate_image(img_bgr, angle)`
Rota la imatge al voltant del seu centre amb interpolació cúbica i `BORDER_REPLICATE` per evitar franges negres.

### `process_crop(crop_bgr)`
Pipeline complet de dues passades per a un crop. Sempre retorna `(chars, aligned, angle, n)` per permetre debugging i visualització dels rebutjats.

In [ ]:
def detect_char_bboxes(thresh, strict):
    """
    Detecta bounding boxes de caràcters sobre una imatge binaritzada.

    Tècnica idèntica al notebook de referència:
      findContours(RETR_TREE, CHAIN_APPROX_SIMPLE) → boundingRect
      → filtre per alçada mediana + aspect ratio

    Paràmetres
    ----------
    thresh : imatge binaritzada uint8 (blanc=text, negre=fons)
    strict : bool
        False → tolerant (passada 1, per estimar l'angle):
                alçada [70%–130%] de la mediana, AR [0.05–1.2]
        True  → estricte (passada 2, segmentació final):
                alçada [85%–115%] de la mediana, AR [0.05–0.95]

    Retorna
    -------
    llista de (x, y, w, h) ordenada per x (esquerra → dreta)
    """
    cnts, _ = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    all_bboxes = [cv2.boundingRect(c) for c in cnts]

    # Mediana d'alçada (ignorem el soroll ≤ 10 px)
    heights = [h for (x, y, w, h) in all_bboxes if h > 10]
    if not heights:
        return []
    h_med = np.median(heights)

    if strict:
        lo_h, hi_h = 0.80, 1.20
        lo_ar, hi_ar = 0.05, 0.90
    else:
        lo_h, hi_h = 0.70, 1.30
        lo_ar, hi_ar = 0.05, 1.20

    bboxes = []
    for (x, y, w, h) in all_bboxes:
        ar = w / float(h)
        if (lo_h * h_med < h < hi_h * h_med) and (lo_ar < ar < hi_ar):
            bboxes.append((x, y, w, h))

    bboxes.sort(key=lambda b: b[0])
    return bboxes


def estimate_skew_angle(bboxes):
    """
    Estima l'angle d'inclinació de la placa per regressió sobre centroides.

    Amb menys de 3 bboxes no hi ha prou senyal: retorna 0.0.
    Si |angle| > ALIGN_ANGLE_MAX probablement és una detecció errònia: retorna 0.0.

    Paràmetres
    ----------
    bboxes : llista de (x, y, w, h)

    Retorna
    -------
    angle : float  (graus, positiu = inclinació horària)
    """
    if len(bboxes) < 3:
        return 0.0

    cxs = np.array([x + w / 2.0 for (x, y, w, h) in bboxes])
    cys = np.array([y + h / 2.0 for (x, y, w, h) in bboxes])

    m, _ = np.polyfit(cxs, cys, 1)
    angle = float(np.degrees(np.arctan(m)))

    if abs(angle) > ALIGN_ANGLE_MAX:
        return 0.0

    return angle


def rotate_image(img_bgr, angle):
    """
    Rota img_bgr `angle` graus al voltant del centre de la imatge.

    Usa cv2.INTER_CUBIC + cv2.BORDER_REPLICATE per evitar franges negres.
    Si angle == 0.0 retorna la mateixa imatge sense copiar.
    """
    if angle == 0.0:
        return img_bgr

    H, W = img_bgr.shape[:2]
    M = cv2.getRotationMatrix2D((W / 2.0, H / 2.0), angle, 1.0)
    return cv2.warpAffine(img_bgr, M, (W, H),
                          flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_REPLICATE)


def prepare_char_for_cnn(char_crop, target_w, target_h):
    """
    Processa el retall d'un caràcter per coincidir amb la distribució de train:
    1. Manté l'aspect ratio encabint-lo en un màxim de 52x52 (64 - 2*margin).
    2. El centra en un llenç de 64x64 (fons negre, text blanc).
    3. En fa un resize final a la mida de la CNN (28x28).
    """
    TRAIN_CANVAS_SZ = 64
    TRAIN_MARGIN = 6
    MAX_CONTENT_SZ = TRAIN_CANVAS_SZ - (2 * TRAIN_MARGIN) # 52 píxels útils
    
    h, w = char_crop.shape[:2]
    if h == 0 or w == 0:
        return np.zeros((target_h, target_w), dtype=np.uint8)
        
    # Calcular factor d'escala basat en la dimensió més llarga (alçada o amplada)
    scale = MAX_CONTENT_SZ / max(h, w)
    nw = max(1, int(round(w * scale)))
    nh = max(1, int(round(h * scale)))
    
    # Redimensionar el caràcter original conservant les proporcions
    char_resized = cv2.resize(char_crop, (nw, nh), interpolation=cv2.INTER_AREA)
    
    # Crear el llenç quadrat de 64x64 (fons negre)
    canvas = np.zeros((TRAIN_CANVAS_SZ, TRAIN_CANVAS_SZ), dtype=np.uint8)
    
    # Centrar el caràcter redimensionat dins el llenç de 64x64
    dx = (TRAIN_CANVAS_SZ - nw) // 2
    dy = (TRAIN_CANVAS_SZ - nh) // 2
    canvas[dy:dy+nh, dx:dx+nw] = char_resized
    
    # Reduir finalment a la mida de la xarxa (CNN_INPUT_W, CNN_INPUT_H -> 28x28)
    return cv2.resize(canvas, (target_w, target_h), interpolation=cv2.INTER_AREA)

def process_crop(crop_bgr):
    """
    Pipeline de dues passades per a un crop de matrícula.

    Passada 1 (tolerant): estima l'angle d'inclinació.
    Rotació: alinea el crop.
    Passada 2 (estricte): segmenta els caràcters finals.

    Retorna
    -------
    chars   : llista de np.ndarray 28×28 uint8, o None si el crop és rebutjat
    aligned : imatge BGR alineada (sempre disponible per a debugging)
    angle   : angle de correcció aplicat en graus
    n       : nombre de caràcters detectats a la passada 2
    """

    # ── PASSADA 1: estimar l'angle (tolerant) ──────────────────────────────────
    gray1   = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    thresh1 = cv2.adaptiveThreshold(gray1, 255,
                                    cv2.ADAPTIVE_THRESH_MEAN_C,
                                    cv2.THRESH_BINARY_INV,
                                    ADAPTIVE_BLOCK, ADAPTIVE_C)
    bboxes1 = detect_char_bboxes(thresh1, strict=False)
    angle   = estimate_skew_angle(bboxes1)

    # ── ROTACIÓ ────────────────────────────────────────────────────────────────
    aligned = rotate_image(crop_bgr, angle)

    # ── PASSADA 2: segmentar sobre la placa alineada (estricte) ───────────────
    gray2   = cv2.cvtColor(aligned, cv2.COLOR_BGR2GRAY)
    thresh2 = cv2.adaptiveThreshold(gray2, 255,
                                    cv2.ADAPTIVE_THRESH_MEAN_C,
                                    cv2.THRESH_BINARY_INV,
                                    ADAPTIVE_BLOCK, ADAPTIVE_C)
    bboxes2 = detect_char_bboxes(thresh2, strict=True)
    n       = len(bboxes2)

    # ── VALIDACIÓ 5–8 ─────────────────────────────────────────────────────────
    if not (N_CHARS_MIN <= n <= N_CHARS_MAX):
        return None, aligned, angle, n

    # ── COPY & RESIZE a 28×28 des de thresh2 ──────────────────────────────────
    chars = [
        prepare_char_for_cnn(thresh2[y:y + h, x:x + w], CNN_INPUT_W, CNN_INPUT_H)
        for (x, y, w, h) in bboxes2
    ]
    return chars, aligned, angle, n


def parse_crop_filename(path):
    """
    Extreu (stem_base, cand_idx) de noms com '{stem}_cand{n}.jpg' o '{stem}_box{n}.png'.
    Retorna (None, None) si el nom no segueix cap dels dos patrons.
    """
    for pattern in [r'^(.+)_cand(\d+)$', r'^(.+)_box(\d+)$']:
        m = re.match(pattern, path.stem)
        if m:
            return m.group(1), int(m.group(2))
    return None, None


print("Funcions definides: detect_char_bboxes, estimate_skew_angle,")
print("                    rotate_image, process_crop, parse_crop_filename")

## Demostració visual sobre un crop d'exemple

La figura de 4 panells mostra:
- **(a)** Crop original amb els bboxes de la **passada 1** (blau) i la **recta de regressió** superposada (taronja). Aquí es veu visualment que la recta captura la inclinació real de la línia de caràcters.
- **(b)** Placa **alineada** amb els bboxes de la **passada 2** (verd). Ara els caràcters queden horitzontals.
- **(c)** Angle estimat i nombre de caràcters detectats.
- **(d)** Els caràcters 28×28 en fila, llestos per a la CNN.

In [ ]:
crop_files = _glob_crops(PROCESSED_DIR)

if not crop_files:
    print(f"No s'han trobat fitxers a '{PROCESSED_DIR}'.")
    print("Executa primer el detector VJ per generar {stem}_cand{n}.jpg a data/processed/.")
else:
    example_path = crop_files[EXAMPLE_INDEX % len(crop_files)]
    crop_bgr     = cv2.imread(str(example_path))

    # ── Passada 1: bboxes tolerants + recta de regressió ─────────────────────
    gray1   = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    thresh1 = cv2.adaptiveThreshold(gray1, 255,
                                    cv2.ADAPTIVE_THRESH_MEAN_C,
                                    cv2.THRESH_BINARY_INV,
                                    ADAPTIVE_BLOCK, ADAPTIVE_C)
    bboxes1 = detect_char_bboxes(thresh1, strict=False)
    angle   = estimate_skew_angle(bboxes1)

    # ── Rotació ───────────────────────────────────────────────────────────────
    aligned = rotate_image(crop_bgr, angle)

    # ── Passada 2: bboxes estrictes sobre la placa alineada ───────────────────
    gray2   = cv2.cvtColor(aligned, cv2.COLOR_BGR2GRAY)
    thresh2 = cv2.adaptiveThreshold(gray2, 255,
                                    cv2.ADAPTIVE_THRESH_MEAN_C,
                                    cv2.THRESH_BINARY_INV,
                                    ADAPTIVE_BLOCK, ADAPTIVE_C)
    bboxes2 = detect_char_bboxes(thresh2, strict=True)
    n2      = len(bboxes2)
    accepted = N_CHARS_MIN <= n2 <= N_CHARS_MAX

    chars_28 = [
        cv2.resize(thresh2[y:y + h, x:x + w], (CNN_INPUT_W, CNN_INPUT_H),
                   interpolation=cv2.INTER_AREA)
        for (x, y, w, h) in bboxes2
    ] if accepted else []

    # ── Construcció de la figura ──────────────────────────────────────────────
    n_chars_row = max(len(chars_28), 1)
    n_cols = max(4, n_chars_row)
    fig = plt.figure(figsize=(max(14, n_cols * 2.5), 7))

    status = f'✓ {n2} caràcters' if accepted else f'✗ REBUTJAT ({n2} caràcters)'
    fig.suptitle(f'{example_path.name}  —  angle={angle:+.2f}°  —  {status}',
                 fontsize=12, fontweight='bold',
                 color='black' if accepted else 'orangered')

    # ── (a) Crop original + bboxes passada 1 + recta de regressió ─────────────
    ax_a = fig.add_subplot(2, n_cols, 1)
    vis_a_bgr = crop_bgr.copy()
    for (x, y, w, h) in bboxes1:
        cv2.rectangle(vis_a_bgr, (x, y), (x + w, y + h), (255, 100, 0), 1)
    ax_a.imshow(cv2.cvtColor(vis_a_bgr, cv2.COLOR_BGR2RGB))
    if len(bboxes1) >= 3:
        cxs = np.array([x + w / 2.0 for (x, y, w, h) in bboxes1])
        cys = np.array([y + h / 2.0 for (x, y, w, h) in bboxes1])
        m_fit, b_fit = np.polyfit(cxs, cys, 1)
        x_line = np.array([0, crop_bgr.shape[1]])
        y_line = m_fit * x_line + b_fit
        ax_a.plot(x_line, y_line, color='orange', linewidth=2, label=f'regressió (m={m_fit:.3f})')
        ax_a.scatter(cxs, cys, color='cyan', s=20, zorder=5)
        ax_a.legend(fontsize=7, loc='upper right')
    ax_a.set_title(f'(a) Passada 1: {len(bboxes1)} bboxes tolerants', fontsize=9)
    ax_a.axis('off')

    # ── (b) Placa alineada + bboxes passada 2 ────────────────────────────────
    ax_b = fig.add_subplot(2, n_cols, 2)
    vis_b = aligned.copy()
    col_b = (0, 200, 0) if accepted else (0, 100, 255)
    for (x, y, w, h) in bboxes2:
        cv2.rectangle(vis_b, (x, y), (x + w, y + h), col_b, 2)
    ax_b.imshow(cv2.cvtColor(vis_b, cv2.COLOR_BGR2RGB))
    ax_b.set_title(f'(b) Passada 2: {n2} bboxes estrictes', fontsize=9,
                   color='green' if accepted else 'orangered')
    ax_b.axis('off')

    # ── (c) Thresh passada 2 ──────────────────────────────────────────────────
    ax_c = fig.add_subplot(2, n_cols, 3)
    ax_c.imshow(thresh2, cmap='gray')
    ax_c.set_title('(c) thresh2 (alineada)', fontsize=9)
    ax_c.axis('off')

    # ── (d) Panell informatiu ─────────────────────────────────────────────────
    ax_d = fig.add_subplot(2, n_cols, 4)
    info = (
        f'angle estimat: {angle:+.2f}°\n'
        f'bboxes P1 (tolerant): {len(bboxes1)}\n'
        f'bboxes P2 (estricte): {n2}\n'
        f'acceptat: {"sí" if accepted else "no"}'
    )
    ax_d.text(0.1, 0.5, info, transform=ax_d.transAxes,
              fontsize=10, verticalalignment='center', fontfamily='monospace')
    ax_d.set_title('(d) Resum', fontsize=9)
    ax_d.axis('off')

    # ── Caràcters 28×28 (fila 2) ──────────────────────────────────────────────
    for i, char_img in enumerate(chars_28):
        ax = fig.add_subplot(2, n_cols, n_cols + i + 1)
        ax.imshow(char_img, cmap='gray', vmin=0, vmax=255)
        ax.set_title(f'char {i}', fontsize=8)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

    print(f"Angle estimat per regressió: {angle:+.2f}°")
    print(f"Bboxes passada 1 (tolerant): {len(bboxes1)}")
    print(f"Bboxes passada 2 (estricte): {n2}")
    print(f"Crop {'ACCEPTAT' if accepted else 'REBUTJAT'}")

## Bucle principal: tots els crops de `data/processed/`

Per a cada `{stem}_cand{n}.jpg`:
1. Aplica el pipeline de dues passades via `process_crop`.
2. Si és acceptat (5–8 caràcters), guarda els caràcters a `data/chars/{stem}_cand{n}_char{i}.png`.
3. Registra les estadístiques i imprimeix una línia per crop.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
crop_files = _glob_crops(PROCESSED_DIR)

if not crop_files:
    print(f"No s'han trobat crops a '{PROCESSED_DIR}'.")
    print("Executa primer el detector VJ per generar {stem}_cand{n}.jpg.")
else:
    print(f"Crops trobats    : {len(crop_files)}")
    print(f"Directori sortida: {OUTPUT_DIR.resolve()}")
    print("-" * 65)

    stats = {'accepted': 0, 'rejected': 0, 'skipped': 0}

    for crop_path in crop_files:
        stem, cand_idx = parse_crop_filename(crop_path)
        if stem is None:
            print(f"  ? {crop_path.name:40s} → nom inesperat (SALTAT)")
            stats['skipped'] += 1
            continue

        crop_bgr_i = cv2.imread(str(crop_path))
        if crop_bgr_i is None:
            print(f"  ! {crop_path.name:40s} → error de lectura (SALTAT)")
            stats['skipped'] += 1
            continue

        chars_i, aligned_i, angle_i, n_i = process_crop(crop_bgr_i)
        accepted_i = chars_i is not None

        if accepted_i:
            # Nom de sortida: {stem_complet}_char{i}.png (traçabilitat total)
            for idx, char_img in enumerate(chars_i):
                out_name = f'{crop_path.stem}_char{idx}.png'
                cv2.imwrite(str(OUTPUT_DIR / out_name), char_img)
            angle_str = f'{angle_i:+.1f}°'
            print(f"  ✓ {crop_path.name:40s} angle={angle_str:>6}  → {n_i} chars guardats")
            stats['accepted'] += 1
        else:
            angle_str = f'{angle_i:+.1f}°'
            print(f"  ✗ {crop_path.name:40s} angle={angle_str:>6}  → {n_i:2d} chars (REBUTJAT)")
            stats['rejected'] += 1

    print("-" * 65)
    total = stats['accepted'] + stats['rejected'] + stats['skipped']
    print(f"\nResum: acceptats={stats['accepted']}  rebutjats={stats['rejected']}  saltats={stats['skipped']}  total={total}")
    print(f"Caràcters desats a: {OUTPUT_DIR.resolve()}")

## Debug: totes les fases del pipeline per a 20 crops

Aquesta secció mostra en detall **totes les fases intermèdies** del pipeline per a una mostra de fins a `N_DEBUG = 20` crops. L'objectiu és facilitar el diagnòstic de casos problemàtics: falsos positius, matrícules mal segmentades, angles erronis, etc.

Per a cada crop es genera una figura amb **2 files**:

| Fila | Panells | Descripció |
|------|---------|------------|
| 1 | **Original** | Crop BGR sense processar |
| 1 | **thresh1 + P1** | Binarització de la passada 1, bboxes tolerants (blau) i recta de regressió (taronja) |
| 1 | **Alineada** | Crop rotat per corregir la inclinació estimada |
| 1 | **thresh2 + P2** | Binarització de la passada 2, bboxes estrictes (verd=acceptat, vermell=rebutjat) |
| 1 | **Info** | Resum numèric: angle, nombre de bboxes a cada passada, resultat |
| 2 | **char 0…N** | Caràcters finals 28×28 px (buits si el crop és rebutjat) |

Els crops es trien aleatòriament (llavor fixa = 42) per garantir diversitat.

In [ ]:
all_crops = _glob_crops(PROCESSED_DIR)

if not all_crops:
    print(f"No s'han trobat crops a '{PROCESSED_DIR}'.")
else:
    # Mostra representativa aleatòria (llavor fixa per reproduïbilitat)
    rng = np.random.default_rng(42)
    if len(all_crops) > N_DEBUG:
        idxs = sorted(rng.choice(len(all_crops), N_DEBUG, replace=False))
        debug_files = [all_crops[i] for i in idxs]
    else:
        debug_files = all_crops

    print(f"Crops totals disponibles : {len(all_crops)}")
    print(f"Crops mostrats al debug  : {len(debug_files)}")
    print("=" * 70)

    for i_dbg, crop_path in enumerate(debug_files):
        crop_bgr = cv2.imread(str(crop_path))
        if crop_bgr is None:
            print(f"  [{i_dbg+1:02d}] ERROR llegint {crop_path.name}")
            continue

        # ── Passada 1 ─────────────────────────────────────────────────────────
        gray1   = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
        thresh1 = cv2.adaptiveThreshold(gray1, 255,
                                        cv2.ADAPTIVE_THRESH_MEAN_C,
                                        cv2.THRESH_BINARY_INV,
                                        ADAPTIVE_BLOCK, ADAPTIVE_C)
        bboxes1 = detect_char_bboxes(thresh1, strict=False)
        angle   = estimate_skew_angle(bboxes1)

        # ── Rotació ───────────────────────────────────────────────────────────
        aligned = rotate_image(crop_bgr, angle)

        # ── Passada 2 ─────────────────────────────────────────────────────────
        gray2   = cv2.cvtColor(aligned, cv2.COLOR_BGR2GRAY)
        thresh2 = cv2.adaptiveThreshold(gray2, 255,
                                        cv2.ADAPTIVE_THRESH_MEAN_C,
                                        cv2.THRESH_BINARY_INV,
                                        ADAPTIVE_BLOCK, ADAPTIVE_C)
        bboxes2 = detect_char_bboxes(thresh2, strict=True)
        n2      = len(bboxes2)
        accepted = N_CHARS_MIN <= n2 <= N_CHARS_MAX


        chars_28 = [
            prepare_char_for_cnn(thresh2[y:y + h, x:x + w], CNN_INPUT_W, CNN_INPUT_H)
            for (x, y, w, h) in bboxes2
        ] if accepted else []

        # ── Layout de la figura ───────────────────────────────────────────────
        n_top  = 5                          # original | thresh1 | aligned | thresh2 | info
        n_char = max(len(chars_28), 1)      # caràcters (mínim 1 per no col·lapsar la fila)
        n_cols = max(n_top, n_char)
        fig    = plt.figure(figsize=(max(14, n_cols * 2.1), 5.5))

        color_t = 'darkgreen' if accepted else 'orangered'
        status  = f'✓ {n2} chars' if accepted else f'✗ {n2} chars (fora de [{N_CHARS_MIN},{N_CHARS_MAX}])'
        fig.suptitle(
            f'[{i_dbg+1:02d}/{len(debug_files)}]  {crop_path.name}'
            f'  |  angle={angle:+.1f}°  |  {status}',
            fontsize=10, fontweight='bold', color=color_t
        )

        # ── Fila 1, panell 1: Original BGR ────────────────────────────────────
        ax = fig.add_subplot(2, n_cols, 1)
        ax.imshow(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
        ax.set_title('Original', fontsize=8, pad=3)
        ax.axis('off')

        # ── Fila 1, panell 2: thresh1 + bboxes P1 + recta de regressió ────────
        ax = fig.add_subplot(2, n_cols, 2)
        vis1 = cv2.cvtColor(thresh1, cv2.COLOR_GRAY2BGR)
        for (x, y, w, h) in bboxes1:
            cv2.rectangle(vis1, (x, y), (x + w, y + h), (255, 120, 0), 1)
        ax.imshow(cv2.cvtColor(vis1, cv2.COLOR_BGR2RGB))
        # Recta de regressió superposada
        if len(bboxes1) >= 3:
            cxs = np.array([bx + bw / 2. for (bx, by, bw, bh) in bboxes1])
            cys = np.array([by + bh / 2. for (bx, by, bw, bh) in bboxes1])
            m_fit, b_fit = np.polyfit(cxs, cys, 1)
            xl = np.array([0, thresh1.shape[1]])
            ax.plot(xl, m_fit * xl + b_fit, color='orange', lw=1.5)
            ax.scatter(cxs, cys, color='cyan', s=12, zorder=5)
        ax.set_title(f'thresh1 + P1 ({len(bboxes1)} bbox)', fontsize=8, pad=3)
        ax.axis('off')

        # ── Fila 1, panell 3: Imatge alineada ────────────────────────────────
        ax = fig.add_subplot(2, n_cols, 3)
        ax.imshow(cv2.cvtColor(aligned, cv2.COLOR_BGR2RGB))
        rot_lbl = f'Alineada ({angle:+.1f}°)' if angle != 0.0 else 'Alineada (0° — ok)'
        ax.set_title(rot_lbl, fontsize=8, pad=3)
        ax.axis('off')

        # ── Fila 1, panell 4: thresh2 + bboxes P2 ────────────────────────────
        ax = fig.add_subplot(2, n_cols, 4)
        vis2 = cv2.cvtColor(thresh2, cv2.COLOR_GRAY2BGR)
        col_p2 = (0, 210, 0) if accepted else (0, 80, 255)
        for (x, y, w, h) in bboxes2:
            cv2.rectangle(vis2, (x, y), (x + w, y + h), col_p2, 2)
        ax.imshow(cv2.cvtColor(vis2, cv2.COLOR_BGR2RGB))
        ax.set_title(f'thresh2 + P2 ({n2} bbox)', fontsize=8, pad=3,
                     color='darkgreen' if accepted else 'orangered')
        ax.axis('off')

        # ── Fila 1, panell 5: Info numèric ───────────────────────────────────
        ax = fig.add_subplot(2, n_cols, 5)
        H_img, W_img = crop_bgr.shape[:2]
        info_lines = [
            f'angle : {angle:+.2f}°',
            f'P1    : {len(bboxes1)} bboxes',
            f'P2    : {n2} bboxes',
            f'rang  : [{N_CHARS_MIN},{N_CHARS_MAX}]',
            f'mida  : {W_img}×{H_img} px',
            '',
            '✓ ACCEPTAT' if accepted else '✗ REBUTJAT',
        ]
        ax.text(0.08, 0.5, '\n'.join(info_lines),
                transform=ax.transAxes, fontsize=8,
                va='center', fontfamily='monospace',
                color='darkgreen' if accepted else 'darkred')
        ax.set_title('Info', fontsize=8, pad=3)
        ax.set_facecolor('#f5f5f5')
        ax.axis('off')

        # ── Fila 2: caràcters 28×28 ───────────────────────────────────────────
        for i_c, char_img in enumerate(chars_28):
            ax = fig.add_subplot(2, n_cols, n_cols + i_c + 1)
            ax.imshow(char_img, cmap='gray', vmin=0, vmax=255)
            ax.set_title(f'c{i_c}', fontsize=7, pad=2)
            ax.axis('off')

        if not chars_28:
            ax = fig.add_subplot(2, n_cols, n_cols + 1)
            ax.text(0.5, 0.5, 'cap caràcter\nacceptat',
                    ha='center', va='center', fontsize=9, color='gray',
                    transform=ax.transAxes)
            ax.axis('off')

        plt.tight_layout(rect=[0, 0, 1, 0.92])
        plt.show()

        print(f"  [{i_dbg+1:02d}] {crop_path.name:35s}"
              f"  angle={angle:+.1f}°  P1={len(bboxes1):2d}  P2={n2:2d}"
              f"  {'✓ OK' if accepted else '✗ REBUTJAT'}")

    print("=" * 70)
    print("Debug completat.")

## Visualització final: matrícules acceptades

Mostrem 3–4 matrícules acceptades. Per a cada una es reexecuta el pipeline complet i es mostren els caràcters 28×28 en una fila.

In [ ]:
char0_files = sorted(OUTPUT_DIR.glob('*_char0.png'))[:50]

if not char0_files:
    print(f"No s'han trobat caràcters a '{OUTPUT_DIR}'.")
    print("Executa primer la cel·la del bucle principal.")
else:
    for char0_path in char0_files:
        prefix = re.sub(r'_char0$', '', char0_path.stem)

        # Deduïm el nom del crop original: el prefix és el stem complet del crop
        # (p.ex. 'eu10_cand2' → busquem 'eu10_cand2.jpg' o 'eu10_cand2.png')
        crop_path_i = None
        for ext in ('.jpg', '.png'):
            candidate = PROCESSED_DIR / f'{prefix}{ext}'
            if candidate.exists():
                crop_path_i = candidate
                break

        if crop_path_i is None:
            print(f"Crop original no trobat: {prefix}")
            continue

        crop_bgr_i = cv2.imread(str(crop_path_i))
        if crop_bgr_i is None:
            continue

        chars_i, aligned_i, angle_i, n_i = process_crop(crop_bgr_i)
        if chars_i is None:
            print(f"  {prefix}: rebutjat ({n_i} caràcters)")
            continue

        n = len(chars_i)
        n_cols = max(n + 2, 4)
        fig = plt.figure(figsize=(max(12, n_cols * 2.2), 5))
        angle_str = f'{angle_i:+.2f}°'
        fig.suptitle(f'{prefix}  |  angle={angle_str}  |  {n} caràcters',
                     fontsize=11, fontweight='bold')

        # Crop original
        ax0 = fig.add_subplot(2, n_cols, 1)
        ax0.imshow(cv2.cvtColor(crop_bgr_i, cv2.COLOR_BGR2RGB))
        ax0.set_title('Original', fontsize=9)
        ax0.axis('off')

        # Placa alineada
        ax1 = fig.add_subplot(2, n_cols, 2)
        ax1.imshow(cv2.cvtColor(aligned_i, cv2.COLOR_BGR2RGB))
        ax1.set_title(f'Alineada ({angle_str})', fontsize=9)
        ax1.axis('off')

        # Caràcters 28×28
        for i, char_img in enumerate(chars_i):
            ax = fig.add_subplot(2, n_cols, n_cols + i + 1)
            ax.imshow(char_img, cmap='gray', vmin=0, vmax=255)
            ax.set_title(f'char {i}', fontsize=8)
            ax.axis('off')

        plt.tight_layout()
        plt.show()